# **Integration with other IEP components**
<font color="#FBB800">Agricultural Cold Chain Access Planning (AgCAP) Tool Output --> SDI [IEP Least-cost, MG potential] </font><br>

-------
## Logic and Rationale for Creating Production, Cooling and PUE Potential

This notebook creates a few **new columns** in `settles_gdf_analyzed`:

- `Fresh Markets Accessibility - class`
- `National Market Accessibility - class`
- `Export Market Accessibility - class`
- `Production_potential`  (deprecated)
- `Cooling_potential`
- `Total_PUE_potential`

The goal is to derive **interpretable, categorical indicators** of productive-use potential per settlement, based on the existing continuous indices in the dataset.

---

### 1. Production_potential (Deprecated)

**Inputs:**

- `Farming Activity` (0–1)
- `Fishing Activity (marine + inland)` (0–1)

**Rationale:**

A settlement can have strong productive potential if **either** farming **or** fishing is strong. We do not want to pre-judge which sector matters more; instead, we capture the **best signal** from the two.

**Steps:**

1. For each of the two columns (`Farming Activity`, `Fishing Activity (marine + inland)`), we classify values into **four evidence-based categories**, using **quantiles** (quartiles):
   - Low  
   - Medium  
   - High  
   - Very High  

   This is done with `pandas.qcut`, which splits the distribution into 4 groups based on the actual data (evidence-based thresholds).

2. For each settlement (row), we look at its **two class labels** (one for farming, one for fishing), convert them to an ordinal scale (Low=0, Medium=1, High=2, Very High=3), and **take the maximum**.

   - If farming is Medium and fishing is High → Production_potential = High  
   - If farming is Low and fishing is Very High → Production_potential = Very High  

3. We convert that ordinal value back into a label and store the result as:

   - `Production_potential` ∈ {Low, Medium, High, Very High}

This ensures that each settlement’s production potential reflects **its strongest productive sector**.

---

### 2. Cooling_potential

**Inputs (cooling-related indices):**

- `Ag Cooling Demand ALL Markets`
- `Fish Cooling Demand ALL Markets`

(All scaled 0–1.)

**Rationale:**

Cooling potential reflects **how strong the demand for cooling is**, across:
- Agricultural products
- Fishery products
- Different market types (export / national / fresh)

If cooling demand for **any** of these streams is high, the settlement is a strong candidate for cold-chain / PUE investments.

**Steps:**

1. For each of the six cooling indices, we again classify values using **quantile-based categories**:
   - Low  
   - Medium  
   - High  
   - Very High  

   This is done separately for each column, so that each indicator uses its own data distribution.

2. For each settlement, we now have up to six class labels (one per cooling index). We convert these labels to ordinal scores (Low=0, Medium=1, High=2, Very High=3) and take the **maximum** across the six.

   - If a settlement is Medium for most, but Very High for Fish Export → Cooling_potential = Very High  
   - If it is Low everywhere → Cooling_potential = Low  

3. We convert this ordinal score back to a label and store the result as:

   - `Cooling_potential` ∈ {Low, Medium, High, Very High}

This ensures that each settlement’s cooling potential reflects **the strongest cooling demand signal** across all ag/fish and market channels.

---

### 3. Total_PUE_potential (Productive Use of Energy Potential)

**Inputs:**

- `Production_potential` (deprecated)
- `Cooling_potential`

**Rationale:**

Overall PUE potential should reflect whether there is **strong productive activity** and/or **strong cooling demand**. If either production or cooling potential is high, the location is interesting from a PUE perspective.

Rather than averaging, we choose a conservative but intuitive approach: **the best of the two** (“if either is strong, treat it as strong PUE potential”).

**Steps:**

1. Convert `Production_potential` and `Cooling_potential` class labels into ordinal scores:
   - Low  → 0  
   - Medium → 1  
   - High → 2  
   - Very High → 3  

2. For each settlement (row), take the **maximum** of the two scores.

   - Production = High (2), Cooling = Medium (1) → PUE_potential score = 2 (High)  
   - Production = Medium (1), Cooling = Very High (3) → PUE_potential score = 3 (Very High)  
   - Both Low → PUE_potential = Low  

3. Convert the final ordinal score back to a label and store it as:

   - `PUE_potential` ∈ {Low, Medium, High, Very High}

---

### Why this approach is useful

- **Evidence-based**: thresholds are derived from the actual data distribution (quantiles), not arbitrary cut-offs.
- **Interpretable**: labels (Low/Medium/High/Very High) are intuitive for planners and developers.
- **Sector-neutral**: production potential respects the strongest of farming or fishing; cooling potential respects the strongest of all cooling demand channels.
- **Decision-ready**: `PUE_potential` provides a single, easy-to-use indicator to filter and prioritise settlements.


# Preparation

## Import packages and functions

In [1]:
import sys
from pathlib import Path

import numpy as np
#import webbrowser
from threading import Timer

# Load the autoreload extension
%load_ext autoreload
%autoreload 2

current_dir = Path.cwd()
project_root = current_dir.parent

sys.path.insert(0, str(project_root))

# Import functions
from scripts.functions import *
from scripts.app import *

### Setting the target coordinate system (Mandatory)

In [2]:
## Coordinate and projection systems
crs_WGS84 = pyproj.CRS("EPSG:4326")    # Originan WGS84 coordinate system
crs_proj = pyproj.CRS("EPSG:32736")    # Projection system for the selected country -- see http://epsg.io/ for more info 

## Read AgCap Generated file

Ensure that the folder `data/processed/input_analyzed` contains the settlements geopackage from the `core_analysis_engine`. This file contains the settlements layer with all AgCap calculated columns.

In [3]:
#Settlements
settles_gdf_analyzed_path= get_multiple_input_file_path(project_root/'data/processed/input_analyzed', allowed_extensions=['gpkg','geojson','fgb','parquet','shp','json','csv'])
print(f'Imported settlement file at this location: {settles_gdf_analyzed_path.relative_to(project_root)}')

#Import settlements
settles_gdf_analyzed = gpd.read_file(settles_gdf_analyzed_path,crs=crs_WGS84)

Imported settlement file at this location: data\processed\input_analyzed\settlements_analyzed_20260423.gpkg


In [4]:
settles_gdf_analyzed.head(2)

,id,Country,Province,District,Posto,Localidade,Urbanization Status,Cluster Area (from IEP),Cluster Area in km2,Population (from IEP),...,Tropical Fruit production,Vegetables production,Tomato production,Fishing Activity (marine + inland),Fishing type,Fish Cooling Demand Export Market,Fish Cooling Demand National Market,Fish Cooling Demand Fresh Markets,Fish Cooling Demand ALL Markets,geometry
0,4,Mozambique,GAZA,CHICUALACUALA,PAFURI,Mbuzi,Low Density Rural,0.521,0.445,88.13,...,0.0,0.0,0.0,NaN,none,NaN,NaN,NaN,NaN,"POLYGON ((31.47039 -22.47395, 31.47112 -22.473..."
1,10,Mozambique,GAZA,CHICUALACUALA,PAFURI,Mbuzi,Low Density Rural,0.650,0.556,117.46,...,0.0,0.0,0.0,NaN,none,NaN,NaN,NaN,NaN,"POLYGON ((31.40961 -22.45221, 31.41106 -22.452..."


### Setting column names, class definitions and functions needed

In [5]:
## Columns for accessibility potential
access_cols = [
    "Fresh Markets Accessibility",
    "National Market Accessibility",
    "Export Market Accessibility",
]

## Columns for production potential
#prod_cols = [
#    "Farming Activity",
#    "Fishing Activity (marine + inland)"
#]

# Cooling potential will consider Export, National, Fresh (Ag + Fish)
#cool_cols = [
#    "Ag Cooling Demand Export Market",
#    "Ag Cooling Demand National Market",
#    "Ag Cooling Demand Fresh Markets",
#    "Fish Cooling Demand Export Market",
#    "Fish Cooling Demand National Market",
#    "Fish Cooling Demand Fresh Markets",
#]

cool_cols = [
    "Ag Cooling Demand ALL Markets",
    "Fish Cooling Demand ALL Markets",
]

# Evidence-based class labels (quartiles)
class_labels = ["Low", "Medium", "High", "Very High"]
class_to_int = {lab: i for i, lab in enumerate(class_labels)}  # Low=0 ... Very High=3
int_to_class = {i: lab for lab, i in class_to_int.items()}

In [6]:
def classify_quantiles(series, labels=class_labels):
    """
    Classify a numeric pandas Series into quantile-based classes.
    By default, splits into 4 quantiles with labels:
    Low, Medium, High, Very High.
    """
    # Work only on non-null values
    non_null = series.dropna()

    if non_null.nunique() == 0:
        # Everything is the same or all NaN; return all NaN classes
        return pd.Series(index=series.index, dtype="object")

    # Use qcut to split into quantiles.
    # duplicates='drop' avoids errors when there are too few unique values.
    cats = pd.qcut(
        non_null,
        q=len(labels),
        labels=labels,
        duplicates="drop"
    )

    # Reinsert into full index, keeping NaNs where original data was NaN
    out = pd.Series(index=series.index, dtype="object")
    out.loc[non_null.index] = cats.astype(str)

    return out

In [7]:
def class_to_score(label):
    """Convert class label to ordinal score; NaNs stay NaN."""
    if pd.isna(label):
        return None
    return class_to_int[label]

#### Calculating Accessibility_potential

In [8]:
# Create class columns for each cooling index
access_class_cols = []
for col in access_cols:
    class_col = f"{col} - class"
    settles_gdf_analyzed[class_col] = classify_quantiles(settles_gdf_analyzed[col])
    access_class_cols.append(class_col)

#### 1. Calculating Production_potential

In [9]:
## Safety check: ensure production columns exist
#missing_prod = [c for c in prod_cols if c not in settles_gdf_analyzed.columns]
#if missing_prod:
#    raise ValueError(f"Missing production columns in dataframe: {missing_prod}")
#
## Create class columns for each production index
#prod_class_cols = []
#for col in prod_cols:
#    class_col = f"{col} - class"
#    settles_gdf_analyzed[class_col] = classify_quantiles(settles_gdf_analyzed[col])
#    prod_class_cols.append(class_col)
#
## Convert class labels to ordinal integers for easy max comparison
#prod_class_int = settles_gdf_analyzed[prod_class_cols].apply(
#    lambda row: max(
#        class_to_int[val] for val in row.dropna()
#    ) if row.notna().any() else None,
#    axis=1
#)
#
## Map back to labels: Low / Medium / High / Very High
#settles_gdf_analyzed["Production_potential"] = prod_class_int.map(int_to_class)

#### 2. Calculating Cooling_potential

In [10]:
# Safety check: ensure cooling columns exist
missing_cool = [c for c in cool_cols if c not in settles_gdf_analyzed.columns]
if missing_cool:
    raise ValueError(f"Missing cooling columns in dataframe: {missing_cool}")

# Create class columns for each cooling index
cool_class_cols = []
for col in cool_cols:
    class_col = f"{col} - class"
    settles_gdf_analyzed[class_col] = classify_quantiles(settles_gdf_analyzed[col])
    cool_class_cols.append(class_col)

# Convert class labels to ordinal integers and pick highest per row
cool_class_int = settles_gdf_analyzed[cool_class_cols].apply(
    lambda row: max(
        class_to_int[val] for val in row.dropna()
    ) if row.notna().any() else None,
    axis=1
)

# Map back to labels
settles_gdf_analyzed["Cooling_potential"] = cool_class_int.map(int_to_class)

#### 3. Calculating Total_PUE_potential

In [11]:
## Convert Production & Cooling potentials to ordinal scores
##prod_scores = settles_gdf_analyzed["Production_potential"].apply(class_to_score) 
#cool_scores = settles_gdf_analyzed["Cooling_potential"].apply(class_to_score)
#
## Take max of the two scores per row
#pue_scores = pd.DataFrame({
#    #"prod": prod_scores,
#    "cool": cool_scores
#}).apply(
#    lambda row: max(
#        v for v in row.values if v is not None
#    ) if any(v is not None for v in row.values) else None,
#    axis=1
#)
#
## Map back to class labels
#settles_gdf_analyzed["PUE_potential"] = pue_scores.map(int_to_class)

### Generating or Updating the Ground-truthing column

In [12]:
## Create a column if it does not exist in the df
if "Ground_Survey" not in settles_gdf_analyzed.columns:
    settles_gdf_analyzed["Ground_Survey"] = "No"

##### Provide the id for the sites to be updated | Replace with "Yes" if surveyed, else leave as "No"

In [13]:
ground_survey_updates = {
    4: "No",
    10: "No",
    13: "No",
}

##### Check if any of the "id" you provided is missing from the df

In [14]:
missing_ids = set(ground_survey_updates.keys()) - set(settles_gdf_analyzed["id"])

if missing_ids:
    print("Warning: these ids were not found in the dataframe:")
    print(missing_ids)
else:
    print("Nothing is missing, you can proceed")

Nothing is missing, you can proceed


##### Run the update

In [15]:
# Make sure we're not modifying a view
settles_gdf_analyzed = settles_gdf_analyzed.copy()

# Update Ground_Survey where id matches the dictionary
settles_gdf_analyzed["Ground_Survey"] = (
    settles_gdf_analyzed["id"]
    .map(ground_survey_updates)               # maps id → Yes/No
    .combine_first(settles_gdf_analyzed["Ground_Survey"])
)

##### Check how many columns have been updated

In [16]:
updated_ids = settles_gdf_analyzed[
    settles_gdf_analyzed["id"].isin(ground_survey_updates.keys())
]
print(f"Updated {len(updated_ids)} rows")

## Show updated value counts
settles_gdf_analyzed["Ground_Survey"].value_counts(dropna=False)

Updated 3 rows


Ground_Survey
No    12321
Name: count, dtype: int64

### Generating crop columns

In [17]:
## Here is a list of all 42 MapSPAM Crop Categories
#cereals = ["Wheat", "Rice", "Maize", "Barley", "Pearl Millet", "Small Millet", "Sorghum", "Other Cereals"]
#roots_and_tubers = ["Cassava", "Potato", "Sweet Potato", "Yams", "Other Roots & Tubers"]
#pulses = ["Bean", "Chickpea", "Cowpea", "Pigeonpea", "Lentil", "Other Pulses"]
#oilseeds = ["Soybean", "Groundnut", "Coconut", "Oil Palm", "Sunflower", "Rapeseed", "Sesame Seed", "Other Oil Crops"]
#sugar_crops = ["Sugarcane", "Sugarbeet"]
#fibers = ["Cotton", "Other Fibre Crops"]
#stimulants = ["Arabica Coffee", "Robusta Coffee", "Cocoa", "Tea", "Tobacco"]
#fruits = ["Banana", "Plantain", "Tropical Fruit", "Temperate Fruit"]
#vegetables = ["Vegetables"]
#other_crops = ["Rest of Crops"]
#
## Master list of all MapSPAM crops
#all_mapspam_crops = (cereals + roots_and_tubers + pulses + oilseeds + sugar_crops + fibers + stimulants + fruits + vegetables + other_crops)

##### Define three main categories of crops

In [18]:
# Staple crops (High calorie, primary food source)
staple = [
    "Wheat",
    "Rice",
    "Maize",
    "Barley",
    "Pearl Millet",
    "Small Millet",
    "Sorghum",
    "Other Cereals",
    "Cassava",
    "Potato",
    "Sweet Potato",
    "Yams",
    "Other Roots & Tubers",
    "Bean",
    "Chickpea",
    "Cowpea",
    "Pigeonpea",
    "Lentil",
    "Other Pulses",
    "Soybean",
    "Groundnut",
    "Banana",
    "Plantain"
]

# Perishable crops (Require cooling/preservation, shorter shelf-life)
perishable = [
    "Vegetables",
    "Tropical Fruit",
    "Temperate Fruit",
    "Banana",
    "Plantain",
    "Cassava",
    "Potato",
    "Sweet Potato",
    "Yams",
    "Other Roots & Tubers"
]

# Cash crops (Grown primarily for profit/industrial use/export)
cash = [
    "Arabica Coffee",
    "Robusta Coffee",
    "Cocoa",
    "Tea",
    "Tobacco",
    "Cotton",
    "Other Fibre Crops",
    "Sugarcane",
    "Sugarbeet",
    "Oil Palm",
    "Coconut",
    "Soybean",
    "Groundnut",
    "Sunflower",
    "Rapeseed",
    "Sesame Seed",
    "Other Oil Crops",
    "Tropical Fruit",
    "Temperate Fruit",
    "Vegetables"
]

In [19]:
settles_gdf_analyzed = settles_gdf_analyzed.copy()

# 1) Find all columns that contain "production" (case-insensitive)
prod_cols = [c for c in settles_gdf_analyzed.columns if "production" in c.lower()]

# If you prefer exact stripping (safer if names are consistent):
def crop_from_col_exact(col: str) -> str:
    return col.replace(" production", "").strip()

# Map column → crop name
crop_name_by_col = {c: crop_from_col_exact(c) for c in prod_cols}

# Make sets for fast membership checks
staple_set = set(staple)
perishable_set = set(perishable)
cash_set = set(cash)

# 2) Pick production columns belonging to each group
staple_cols = [c for c in prod_cols if crop_name_by_col[c] in staple_set]
perishable_cols = [c for c in prod_cols if crop_name_by_col[c] in perishable_set]
cash_cols = [c for c in prod_cols if crop_name_by_col[c] in cash_set]

# 3) Create the 3 new summary columns
settles_gdf_analyzed["Staple_Crop Production (t/y)"] = (settles_gdf_analyzed[staple_cols].sum(axis=1) if staple_cols else 0)
settles_gdf_analyzed["Perishable_Crop Production (t/y)"] = (settles_gdf_analyzed[perishable_cols].sum(axis=1) if perishable_cols else 0)
settles_gdf_analyzed["Cash Crop Production (t/y)"] = (settles_gdf_analyzed[cash_cols].sum(axis=1) if cash_cols else 0)

In [20]:
# Quick check of what was matched
print("Production columns found:", len(prod_cols))
print("Staple columns matched:", staple_cols)
print("Perishable columns matched:", perishable_cols)
print("Cash columns matched:", cash_cols)

Production columns found: 18
Staple columns matched: ['Banana production', 'Cassava production', 'Maize production', 'Other Roots & Tubers production', 'Plantain production', 'Potato production', 'Rice production', 'Sorghum production', 'Sweet Potato production']
Perishable columns matched: ['Banana production', 'Cassava production', 'Other Roots & Tubers production', 'Plantain production', 'Potato production', 'Sweet Potato production', 'Temperate Fruit production', 'Tropical Fruit production', 'Vegetables production']
Cash columns matched: ['Cotton production', 'Sugarcane production', 'Temperate Fruit production', 'Tobacco production', 'Tropical Fruit production', 'Vegetables production']


#### Develop files for cooling potential scenarios (Post-Harvest & Fishing)

In [21]:
ag_class_col   = 'Ag Cooling Demand ALL Markets - class'
#ag_class_col   = f'{cool_cols[0]} - class'
fish_class_col = 'Fish Cooling Demand ALL Markets - class'
#fish_class_col = f'{cool_cols[1]} - class'

# --- Scenario 1: Post-Harvest Cooling (Ag) ---
postharvest_df = settles_gdf_analyzed.copy()
postharvest_df['PUE_potential'] = postharvest_df[ag_class_col]
postharvest_df = postharvest_df.dropna(subset=['PUE_potential'])

# --- Scenario 2: Fishing Cooling ---
fishing_df = settles_gdf_analyzed.copy()
fishing_df['PUE_potential'] = fishing_df[fish_class_col]
fishing_df = fishing_df.dropna(subset=['PUE_potential'])

# --- UTF-8 compatibility and null filling ---
def prepare_for_export(df):
    for col in df.columns:
        if col == 'geometry':
            continue
        if df[col].dtype == 'object':
            df[col] = df[col].fillna('').apply(
                lambda x: x.encode('utf-8', errors='replace').decode('utf-8')
                if isinstance(x, str) else x
            )
        else:
            df[col] = df[col].fillna(0)
    return df

postharvest_df = prepare_for_export(postharvest_df)
fishing_df     = prepare_for_export(fishing_df)

# Export results for integration

In [22]:
## ## Export to geopackage
#settles_gdf_analyzed.to_file(project_root/f"data/processed/input_analyzed/settlements_analyzed_for_IEP_{datetime.now().strftime('%Y%m%d')}.gpkg", driver="GPKG")

## To csv in case that is needed
#settles_gdf_analyzed.drop(columns="geometry").to_csv(project_root/f"data/processed/input_analyzed/settlements_analyzed_for_IEP_{datetime.now().strftime('%Y%m%d')}.csv", index=False, encoding="utf-8-sig")

print(f"PostHarvest rows exported: {len(postharvest_df)}")
postharvest_df.drop(columns='geometry', errors='ignore').to_csv(project_root/f"data/processed/input_analyzed/PostHarvest_Cooling_Potential_Scenario_{datetime.now().strftime('%Y%m%d')}.csv", index=False, encoding="utf-8")

print(f"Fishing rows exported:     {len(fishing_df)}")
fishing_df.drop(columns='geometry', errors='ignore').to_csv(project_root/f"data/processed/input_analyzed/Fishing_Cooling_Potential_Scenario_{datetime.now().strftime('%Y%m%d')}.csv", index=False, encoding="utf-8")

PostHarvest rows exported: 12242
Fishing rows exported:     4154
